# Bi-LSTM with GloVe

Train and evaluate the LSTM ABSA model. GloVe is optional; without it we use random embeddings.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_processed_splits
from src.preprocess import encode_labels
from src.models.lstm_model import ABSADataset, BiLSTMClassifier, build_vocab_and_weights
from src.evaluate import compute_all_metrics, plot_confusion_matrix
from src.utils import set_seed, get_device

In [ ]:
set_seed(42)
device = get_device(True)
processed_dir = ROOT / "data" / "processed"
if (processed_dir / "train.csv").exists():
    train_df, val_df, test_df = load_processed_splits(str(processed_dir))
else:
    train_df = pd.DataFrame({
        "sentence": ["Great food and service."] * 40,
        "aspect_term": ["food", "service"] * 20,
        "polarity": ["positive", "negative"] * 20,
    })
    val_df = test_df = train_df.iloc[:10]
train_df["label"] = encode_labels(train_df["polarity"])
test_df["label"] = encode_labels(test_df["polarity"])
word2idx, emb_weights = build_vocab_and_weights(
    train_df["sentence"].tolist(), train_df["aspect_term"].tolist(),
    glove_path=None, embedding_dim=100,
)
max_len = 128
batch_size = 16
train_ds = ABSADataset(train_df["sentence"].tolist(), train_df["aspect_term"].tolist(), train_df["label"].tolist(), word2idx, max_len)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_ds = ABSADataset(test_df["sentence"].tolist(), test_df["aspect_term"].tolist(), test_df["label"].tolist(), word2idx, max_len)
test_loader = DataLoader(test_ds, batch_size=batch_size)
print("Vocab size:", len(word2idx))

In [ ]:
model = BiLSTMClassifier(
    vocab_size=len(word2idx), embedding_dim=100, hidden_size=128, num_layers=2,
    num_classes=4, dropout=0.5, embedding_weights=emb_weights,
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
print("Training done.")

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for x, _ in test_loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().numpy().tolist())
metrics = compute_all_metrics(test_df["label"].tolist(), preds)
print("Metrics:", metrics)
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(test_df["label"].tolist(), preds, save_path=ROOT / "results" / "figures" / "notebook_lstm_cm.png", title="LSTM Confusion Matrix")